<a href="https://colab.research.google.com/github/thiranesh/Thiranesh-codeboosters-2026/blob/main/day-5/Day_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#import all required libraies

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [4]:
df = pd.read_csv('student_performance.csv')
print(f'Dataset loaded: {df.shape[0]} student, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print(f'\nMissing values: {df.isnull().sum().values.sum()}')
print('\nFirst 5 rows: ')
df.head()

Dataset loaded: 30 student, 13 columns
Columns: ['student_id', 'name', 'age', 'gender', 'department', 'semester', 'math_score', 'science_score', 'english_score', 'programming_score', 'attendance_percentage', 'city', 'admission_year']

Missing values: 0

First 5 rows: 


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [5]:
df_ml = df.copy()
le_gender = LabelEncoder()
df_ml['gender_encoded'] = le_gender.fit_transform(df_ml['gender'])
print(f'Gender encoding:{dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_)))}')
le_dept = LabelEncoder()
df_ml['department_encoded'] = le_dept.fit_transform(df_ml['department'])
print(f'Department encoding:{dict(zip(le_dept.classes_,le_dept.transform(le_dept.classes_)))}')
print('\nNew Columns added:gender_encoded, department_encoded')
df_ml[['gender','gender_encoded','department','department_encoded']].head(5)

Gender encoding:{'Female': np.int64(0), 'Male': np.int64(1)}
Department encoding:{'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}

New Columns added:gender_encoded, department_encoded


,gender,gender_encoded,department,department_encoded
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [6]:
print(df["programming_score"].describe())

count    30.000000
mean     67.600000
std      21.041175
min      38.000000
25%      49.250000
50%      66.000000
75%      88.750000
max      97.000000
Name: programming_score, dtype: float64


In [7]:
feature_cols = [
    'math_score',
    'science_score',
    'english_score',
    'attendance_percentage',
    'gender_encoded',
    'department_encoded'
]

X = df_ml[feature_cols]
y = df_ml['programming_score']

print(f'Feature matrix X shape: {X.shape}(students x features)')
print(f'Target vector y shape: {y.shape}(one score per student)')
print(f'\nFeature columns: {feature_cols}')
print(f'Target column: {y.name}')
print(f'\nTarget range: {y.min()} to {y.max()} (mean:{y.mean():.1f})')

Feature matrix X shape: (30, 6)(students x features)
Target vector y shape: (30,)(one score per student)

Feature columns: ['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_encoded', 'department_encoded']
Target column: programming_score

Target range: 38 to 97 (mean:67.6)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Total Students:{len(X)}')
print(f'Training students:{len(X_train)}({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing students:{len(X_test)}({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTraining target range: {y_train.min()} - {y_train.max()}')
print(f'Testing target range: {y_test.min()} - {y_test.max()}')

Total Students:30
Training students:24(80%)
Testing students:6(20%)

Training target range: 38 - 96
Testing target range: 39 - 97


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Features scaled Successfully!')
print(f'Training feature mean(should be = 0): {X_train_scaled.mean():.2f}')
print(f'Testing feature std(should be = 1): {X_test_scaled.std():.2f}')

print('Note: Scaler was fit on training data only - prevents data leakage')

Features scaled Successfully!
Training feature mean(should be = 0): -0.00
Testing feature std(should be = 1): 1.21
Note: Scaler was fit on training data only - prevents data leakage


In [13]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred = lr_model.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, y_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
lr_r2 = r2_score(y_test, y_pred)
print('===Model 1: Linear Regression===')
print(f'MAE: {lr_mae:.2f}(predictions are off by {lr_mae:.1f} points on average)')
print(f'RMSE: {lr_rmse:.2f}(error with penalty for large mistakes)')
print(f'R-square: {lr_r2:.4f}({lr_r2*100:.1f}% of programming score variation explained)')
print()
print('Leaned coefficients I(importance of each feature)')
for feat, coef in zip(feature_cols, lr_model.coef_):
    print(f'{feat:<28}: {coef:+.3f}')
print(f'{"bias (intercept)":<28}:{lr_model.intercept_:.3f}')

===Model 1: Linear Regression===
MAE: 9.37(predictions are off by 9.4 points on average)
RMSE: 11.47(error with penalty for large mistakes)
R-square: 0.7356(73.6% of programming score variation explained)

Leaned coefficients I(importance of each feature)
math_score                  : +20.256
science_score               : +7.612
english_score               : +2.396
attendance_percentage       : -12.886
gender_encoded              : -0.397
department_encoded          : -0.449
bias (intercept)            :68.042


In [16]:
dt_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)
dt_model.fit(X_train_scaled, y_train)
dt_pred = dt_model.predict(X_test_scaled)

dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
dt_r2 = r2_score(y_test, dt_pred)

print('===Model 2: Decision Tree(max_depth=5)===')
print(f'MAE: {dt_mae:.2f}')
print(f'RMSE :{dt_rmse:.2f}')
print(f'R-square: {dt_r2:.4f}')

===Model 2: Decision Tree(max_depth=5)===
MAE: 10.75
RMSE :16.08
R-square: 0.4803
